# Candidate recovery metric: self-reported physical health

Patient-reported outcomes are the only candidate here that measures recovery as the patient
experiences it, and they are the instruments clinical readers find most credible. The obstacle is
timing rather than validity: All of Us administers surveys on an enrolment cadence, not relative
to anyone's surgery, so for most participants there may be no pre/post pair bracketing the index
date at all.

This notebook finds the relevant survey questions by concept rather than by a hard-coded ID list,
counts who answered them, and -- the decisive question -- measures when those answers fall
relative to surgery.

**Governance.** Every query aggregates inside BigQuery and returns counts,
quantiles or correlations only, so no participant-level rows enter the notebook. Unsuppressed
figures stay git-ignored while `*_suppressed.png` is committable, and the small-cell threshold
must be confirmed against current All of Us policy before anything is published.

# 1-setup

### Datasets, cohort SQL and output directories

`srp_common` resolves the pinned CDR and cohort index, creates this notebook's git-ignored
figure and cache directories, and builds the cohort plus one index date per patient. The
metric-specific CTE is defined here too, so the whole SQL definition this notebook runs on sits
in one place.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas

_root = Path.cwd().resolve()
while not (_root / "srp_common.py").exists():
    if _root.parent == _root:
        raise RuntimeError(
            "Repo root not found: no srp_common.py at or above "
            f"{Path.cwd().resolve()}. Run this notebook from inside the repository."
        )
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import srp_common
from srp_common import CATEGORICAL, INK, INK_MUTED, SERIES, SUPPRESS_THRESHOLD

cfg = srp_common.init("survey_physical_health")
CDR = cfg.CDR
FIGURES, CACHE_DIR = cfg.FIGURES, cfg.CACHE_DIR
cached_gbq = cfg.cached_gbq

# Which of a patient's surgery-days becomes the index date is still open --
# index_surgery/surgery_day_distribution.ipynb is what settles it. "first" is used
# until then because it is the only rule no post-operative event can influence, and
# every result below is stamped with the rule so a later change is visible.
INDEX_RULE = "first"
WINDOWS = srp_common.DEFAULT_WINDOWS

# Survey answers live in `observation` with PPI-vocabulary concepts. The concept IDs are
# not hard-coded, because they differ across releases and a stale ID silently returns
# nothing; section 2 discovers them by name instead and prints what it found.
SURVEY_NAME_PATTERNS = [
    "%general health%",
    "%overall health%",
    "%physical health%",
    "%quality of life%",
]

METRIC_CTE = ""

CTES = cfg.cohort_ctes(INDEX_RULE, extra=METRIC_CTE)

print(f"index rule : {INDEX_RULE}")
print("windows    : " + ", ".join(f"{k} [{lo}, {hi}]" for k, (lo, hi) in WINDOWS.items()))

# Fail here, not several minutes into a BigQuery run, if the CDR does not have
# what this notebook reads. cohort_ctes checks its own tables separately.
cfg.require_columns({
    "concept": ["concept_id", "concept_name", "concept_code", "vocabulary_id"],
    "observation": ["person_id", "observation_concept_id", "observation_date"],
})


# 2-coverage

### Which patient-reported health questions exist, and how many cohort members answered them?

Discovery is by concept name within the PPI vocabulary, so a release that renumbers its survey
concepts still resolves. Read the printed list before trusting anything below -- the patterns are
broad on purpose and will catch questions that are not the intended instrument.

In [ ]:
pattern_clause = " OR ".join(
    f"LOWER(c.concept_name) LIKE '{p}'" for p in SURVEY_NAME_PATTERNS
)

questions_sql = cfg.cohort_ctes(INDEX_RULE) + f"""
    SELECT
        c.concept_id,
        c.concept_name,
        c.concept_code,
        COUNT(DISTINCT o.person_id) AS n_patients,
        COUNT(*)                    AS n_answers
    FROM `{CDR}`.observation AS o
    JOIN index_surgery AS i ON i.person_id = o.person_id
    JOIN `{CDR}`.concept AS c ON c.concept_id = o.observation_concept_id
    WHERE c.vocabulary_id = 'PPI'
      AND ({pattern_clause})
    GROUP BY c.concept_id, c.concept_name, c.concept_code
    HAVING COUNT(DISTINCT o.person_id) >= {SUPPRESS_THRESHOLD}
    ORDER BY n_patients DESC
"""

survey_questions = cached_gbq(questions_sql, "survey_health_questions")

if survey_questions.empty:
    print("No PPI question matching the patterns was answered by at least "
          f"{SUPPRESS_THRESHOLD} cohort members. Widen SURVEY_NAME_PATTERNS or record "
          "this metric as unavailable.")
else:
    print(survey_questions.to_string(index=False))

SURVEY_CONCEPT_IDS = [int(v) for v in survey_questions["concept_id"]]

### How many cohort patients answered any of them, against the cohort denominator?

This is the ceiling on a survey-based outcome before timing is even considered. It is reported
against the same denominator as every other metric so the scorecard rows are comparable.

In [ ]:
# An empty list would make the IN clause a syntax error, so it degrades to NULL --
# which matches nothing and yields an honest zero rather than a failed query.
CONCEPT_ID_LIST = ", ".join(str(i) for i in SURVEY_CONCEPT_IDS) or "NULL"

coverage_sql = cfg.cohort_ctes(INDEX_RULE) + f"""
    SELECT
        (SELECT COUNT(*) FROM cohort) AS cohort_patients,
        COUNT(DISTINCT o.person_id)   AS patients_with_any_answer,
        COUNT(*)                      AS n_answers
    FROM `{CDR}`.observation AS o
    JOIN index_surgery AS i ON i.person_id = o.person_id
    WHERE o.observation_concept_id IN ({CONCEPT_ID_LIST})
"""

cov = cached_gbq(coverage_sql, "survey_health_coverage").iloc[0]
cohort_patients = int(cov["cohort_patients"])
patients_with_any_answer = int(cov["patients_with_any_answer"])

print(f"cohort patients            : {cohort_patients:>9,}")
print(f"  ... answered >=1 question: {patients_with_any_answer:>9,}"
      f"   ({patients_with_any_answer / cohort_patients:6.1%} of cohort)")
print(f"answers contributed        : {int(cov['n_answers']):>9,}")

# 3-timing

### When do those answers fall relative to the surgery?

This is the question that decides the metric: a survey-based recovery outcome needs at least one
answer before the index date and one after it, for the same patient. Answers are bucketed into
90-day bands because survey dates cluster on enrolment waves rather than spreading evenly.

In [ ]:
timing_sql = cfg.cohort_ctes(INDEX_RULE) + f"""
    , answers AS (
        SELECT
            o.person_id,
            DATE_DIFF(o.observation_date, i.index_date, DAY) AS rel_day
        FROM `{CDR}`.observation AS o
        JOIN index_surgery AS i ON i.person_id = o.person_id
        WHERE o.observation_concept_id IN ({CONCEPT_ID_LIST})
          AND o.observation_date IS NOT NULL
    )
    SELECT
        CAST(FLOOR(rel_day / 90) AS INT64) * 90 AS quarter_start_day,
        COUNT(DISTINCT person_id)               AS n_patients
    FROM answers
    WHERE rel_day BETWEEN -1825 AND 1825
    GROUP BY quarter_start_day
    ORDER BY quarter_start_day
"""

survey_timing = cached_gbq(timing_sql, "survey_health_timing")
survey_timing["rel_day"] = survey_timing["quarter_start_day"]

srp_common.plot_rel_day_curve(
    cfg, survey_timing, "n_patients",
    title="Patients answering a health question, by 90-day band around surgery",
    ylabel="patients with an answer in the band", suppress=True,
    show_windows=False, save_as="survey_timing")

### How many patients have an answer both before and after the surgery?

Only these patients can contribute a change score, which is what a survey-based recovery outcome
actually is. A count below the threshold here ends the metric's candidacy regardless of how good
the instrument is.

In [ ]:
paired_sql = cfg.cohort_ctes(INDEX_RULE) + f"""
    , answers AS (
        SELECT
            o.person_id,
            DATE_DIFF(o.observation_date, i.index_date, DAY) AS rel_day
        FROM `{CDR}`.observation AS o
        JOIN index_surgery AS i ON i.person_id = o.person_id
        WHERE o.observation_concept_id IN ({CONCEPT_ID_LIST})
          AND o.observation_date IS NOT NULL
    ),
    per_person AS (
        SELECT
            person_id,
            COUNTIF(rel_day BETWEEN -730 AND -1) AS answers_before,
            COUNTIF(rel_day BETWEEN 1 AND 365)   AS answers_after_1y,
            COUNTIF(rel_day BETWEEN 30 AND 365)  AS answers_after_recovery_window
        FROM answers
        GROUP BY person_id
    )
    SELECT
        COUNT(*)                                                        AS patients_with_answers,
        COUNTIF(answers_before > 0)                                     AS with_pre,
        COUNTIF(answers_after_1y > 0)                                   AS with_post,
        COUNTIF(answers_before > 0 AND answers_after_1y > 0)             AS paired,
        COUNTIF(answers_before > 0 AND answers_after_recovery_window > 0) AS paired_after_day_30
    FROM per_person
"""

survey_paired = cached_gbq(paired_sql, "survey_health_paired").iloc[0]
paired_patients = int(survey_paired["paired"])

for label in ["patients_with_answers", "with_pre", "with_post", "paired",
              "paired_after_day_30"]:
    n = int(survey_paired[label])
    flag = "  << below threshold, do not report" if 0 < n < SUPPRESS_THRESHOLD else ""
    print(f"{label:24s} {n:>9,}   ({n / cohort_patients:6.1%} of cohort){flag}")

print()
print("--> 'paired' is the analysable population for a change-score outcome. If it is a "
      "small fraction of the cohort, this metric is a secondary at best.")

# 4-verdict

### What is the verdict for this metric?

The audit criteria are recorded as one row in the shared task cache, which is what
`metric_comparison.ipynb` reads to build the cross-metric scorecard. Nothing here is a
participant-level value: the row is counts, fractions and hand-entered judgements.

In [ ]:
srp_common.write_scorecard(
    cfg,
    family="survey",
    source_table="observation (PPI)",
    index_rule=INDEX_RULE,
    outcome_type="ordinal",
    cohort_patients=cohort_patients,
    patients_with_metric=patients_with_any_answer,
    patients_analysable=paired_patients,
    coverage_share=paired_patients / cohort_patients,
    horizon_days=365,
    informative_missingness="unclear",
    leakage_risk="low",
    notes=("Timing, not validity, is the constraint: All of Us surveys are administered on an "
           "enrolment cadence rather than relative to surgery, so a pre/post pair is incidental. "
           "Concept IDs are discovered by name each run rather than hard-coded."),
)